# Abstract

- Goal: Train and tune basic classification models to reach good accuracy score

- Dataset: [IMDB Dataset of 50K Movie Reviews (Kaggle)](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews)

- Project Details:

    I'm used base classification models such like Logistic Regression, Decision Tree, Random Forest, Gradient Boosting (XGB). Also i make form for prediction your own text.
- Best result: 0.839 (Accuracy Score of Logistic Regression)

- Sections:
    - [Imports](#Imports)
    - [Dataset](#Dataset)
    - [Modeling](#Modeling)
        - [Logistic Regression](#Logistic-Regression)
        - [Decision Tree](#Decision-Tree)
        - [Random Forest](#Random-Forest)
        - [Gradient Boost](#Gradient-Boost)
    - [Prediction](#Prediction)
        - [Setup Form](#Setup-Form)
        - [Prediction Form](#Prediction-Form)

## Problems & Solutions
- Problem 1: All Models are Overfitted

    **Symptoms:**

    - All models have great train accuracy but low validation accuracy

    **Root Cause:**

    - Big model input size
    - Trash in input text

    **Solution:**

    - TfidfVectorizer's `max_features` parameter is too big for this task. I changed it from 5000 -> 450.
    - Add text preprocessing function



# Download & Install Dependencies

In [ ]:
# Dataset Downloading
!mkdir data
!curl -L -o ./data/dataset.zip https://www.kaggle.com/api/v1/datasets/download/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
!unzip ./data/dataset.zip -d ./data
!rm ./data/dataset.zip
!mv ./data/IMDB\ Dataset.csv ./data/imdb_dataset.csv

In [ ]:
# For colab users (local library files download)
!curl -L -o ./setup_text_preprocessing.py https://raw.githubusercontent.com/jsonmen/bias-and-variance/refs/heads/main/SentimentAnalysis/setup_text_preprocessing.py
!curl -L -o ./text_preprocessing.py https://raw.githubusercontent.com/jsonmen/bias-and-variance/refs/heads/main/SentimentAnalysis/text_preprocessing.py

# Imports

In [2]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, KFold
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import ipywidgets as widgets
from IPython.display import clear_output, display
from text_preprocessing import text_preprocessing
from tqdm import tqdm

# Dataset

In [9]:
df = pd.read_csv("./data/imdb_dataset.csv")
train_dataset, test_dataset = df.iloc[:10000].copy(), df.iloc[10000:12000].copy()

In [4]:
sentiment2label = {"positive": 1, "negative": 0}
label2sentiment = {1: "positive", 0: "negative"}

In [10]:
train_dataset["preprocessedText"] = train_dataset["review"].apply(text_preprocessing)
test_dataset["preprocessedText"] = test_dataset["review"].apply(text_preprocessing)

In [11]:
vectorizer = TfidfVectorizer(
    max_features=450,
    min_df=5,
    max_df=0.7,
)
X = vectorizer.fit_transform(train_dataset["preprocessedText"])
y = train_dataset['sentiment'].map(sentiment2label)

X_test = vectorizer.transform(test_dataset["preprocessedText"])
y_test = test_dataset['sentiment'].map(sentiment2label)

vocab = vectorizer.get_feature_names_out()
print(f"{X.shape=}, {y.shape=}, {X_test.shape=}, {y_test.shape=}")
print(f"Number of unique words: {len(vocab)}")
print(vocab)

X.shape=(10000, 450), y.shape=(10000,), X_test.shape=(2000, 450), y_test.shape=(2000,)
Number of unique words: 450
['able' 'about' 'absolutely' 'act' 'acting' 'action' 'actor' 'actress'
 'actually' 'add' 'after' 'again' 'against' 'age' 'agent' 'all' 'almost'
 'along' 'already' 'also' 'although' 'always' 'american' 'an' 'another'
 'any' 'anyone' 'anything' 'appear' 'around' 'art' 'as' 'ask' 'at'
 'attempt' 'audience' 'away' 'awful' 'back' 'bad' 'base' 'beautiful'
 'because' 'become' 'before' 'begin' 'believe' 'between' 'big' 'bit'
 'black' 'book' 'boring' 'both' 'boy' 'break' 'bring' 'brother' 'budget'
 'buy' 'call' 'camera' 'can' 'car' 'care' 'case' 'cast' 'certainly'
 'change' 'character' 'child' 'cinema' 'classic' 'close' 'come' 'comedy'
 'comment' 'completely' 'consider' 'could' 'couple' 'course' 'create'
 'cut' 'dark' 'day' 'dead' 'deal' 'death' 'decide' 'definitely' 'despite'
 'dialogue' 'didst' 'die' 'different' 'direct' 'direction' 'director' 'do'
 'done' 'down' 'drama' 'during'

# Modeling

## Logistic Regression

In [12]:
### Model Parameter Searching
lr_ = LogisticRegression()

parameters = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']  # 'liblinear' supports both l1 and l2 penalties
}

lr_grid = GridSearchCV(lr_, parameters, cv=5, scoring='accuracy')
lr_grid.fit(X, y)

print("Best Model Score:", lr_grid.best_score_)
print("Best Model Parameters:", lr_grid.best_params_)

Best Model Score: 0.8352
Best Model Parameters: {'C': 1, 'penalty': 'l1', 'solver': 'liblinear'}


In [13]:
### Test Model Performance
lr = LogisticRegression(**lr_grid.best_params_)
lr.fit(X, y)
y_pred = lr.predict(X_test)
print("Model Test Score:", accuracy_score(y_pred, y_test))

Model Test Score: 0.839


## Decision Tree

In [14]:
### Model Parameter Searching
dt_ = DecisionTreeClassifier()

parameters = {
    'max_depth': [3, 5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

dt_grid = RandomizedSearchCV(dt_, parameters, cv=5, scoring='accuracy', n_iter=50, n_jobs=-1)
dt_grid.fit(X, y)

print("Best Model Score:", dt_grid.best_score_)
print("Best Model Parameters:", dt_grid.best_params_)

Best Model Score: 0.7215
Best Model Parameters: {'min_samples_split': 10, 'min_samples_leaf': 2, 'max_depth': 10, 'criterion': 'gini'}


In [15]:
### Test Model Performance
dt = DecisionTreeClassifier(**dt_grid.best_params_)
dt.fit(X, y)
y_pred = dt.predict(X_test)
print("Model Test Score:", accuracy_score(y_pred, y_test))

Model Test Score: 0.718


## Random Forest

In [16]:
### Model Parameter Searching
rf_ = RandomForestClassifier(random_state=42)

parameters = {
    'n_estimators': [50, 100, 200],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'] 
}

rf_grid =  RandomizedSearchCV(rf_, parameters, cv=5, scoring='accuracy', n_iter=50, n_jobs=-1)
rf_grid.fit(X, y)

print("Best Model Score:", rf_grid.best_score_)
print("Best Model Parameters:", rf_grid.best_params_)

Best Model Score: 0.8209000000000002
Best Model Parameters: {'n_estimators': 200, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 20}


In [17]:
### Test Model Performance
rf = RandomForestClassifier(random_state=42, **rf_grid.best_params_)
rf.fit(X, y)
y_pred = rf.predict(X_test)
print("Model Test Score:", accuracy_score(y_pred, y_test))

Model Test Score: 0.816


## Gradient Boost

In [18]:
### Model Parameter Searching
xgb_ = XGBClassifier(random_state=42)

parameters = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10],
    'learning_rate': [0.01, 0.1, 0.3],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0]
}

xgb_grid = RandomizedSearchCV(xgb_, parameters, cv=5, scoring='accuracy', n_iter=20, n_jobs=-1)
xgb_grid.fit(X, y)

print("Best Model Score:", xgb_grid.best_score_)
print("Best Model Parameters:", xgb_grid.best_params_)

Best Model Score: 0.8158000000000001
Best Model Parameters: {'subsample': 0.8, 'n_estimators': 100, 'max_depth': 10, 'learning_rate': 0.1, 'colsample_bytree': 1.0}


In [19]:
### Test Model Performance
xgb = XGBClassifier(random_state=42, **xgb_grid.best_params_)
xgb.fit(X, y)
y_pred = xgb.predict(X_test)
print("Model Test Score:", accuracy_score(y_pred, y_test))

Model Test Score: 0.819


# Prediction

## Setup Form

In [20]:
text_field = widgets.Text(
    value='',
    placeholder='Type something',
    description='Text:',
    disabled=False   
)
submit_button = widgets.Button(description="Analyze Sentiment")
output = widgets.Output()
def form_fn(b):
    text = text_field.value
    preprocessed_text = vectorizer.transform([text_preprocessing(text)])
    sentiment_label = lr.predict(preprocessed_text)
    sentiment_word = label2sentiment[sentiment_label[0]]
    with output:
        clear_output()
        print(f"Your text has a {sentiment_word} sentiment")
submit_button.on_click(form_fn)
form = widgets.VBox([text_field, submit_button, output])

## Prediction Form

In [23]:
display(form)